In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from functools import partial
import matplotlib.pyplot as plt
import scipy.io as io
import sys
import numpy as np
from tqdm import tqdm
import gc

global_path = "/home/ids/edabier/HSU"
sys.path.append(f"{global_path}/SS-HSU_benchmark")

from src.utils import utils, losses, plots, extractor
from src.models import models, upsamplers, foundation_models as rsfm, unmixers as unmx

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(dev)
print(f"Using device: {dev}")

/home/ids/edabier/miniconda3/envs/hsu-latest/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


/home/ids/edabier/miniconda3/envs/hsu-latest/lib/python3.14/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/home/ids/edabier/miniconda3/envs/hsu-latest/lib/python3.14/site-packages/timm/models/helpers.py:7: FutureWarning: Importing from timm.models.helpers is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/home/ids/edabier/miniconda3/envs/hsu-latest/lib/python3.14/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


In [2]:
dataset = "urban"
data = io.loadmat(f"{global_path}/SS-HSU_benchmark/datasets/{dataset}.mat")
Y_flat = torch.tensor(data["Y"], dtype=torch.float)
A_flat = torch.tensor(data["A"], dtype=torch.float)
E_init = torch.tensor(data["E"], dtype=torch.float)
B, c, N = E_init.shape[0], E_init.shape[1], Y_flat.shape[1]

Y_init = utils.oneD_to_2d(Y_flat)
A_init = utils.oneD_to_2d(A_flat)
H = Y_init.shape[-1]
Y_init, A_init = rsfm.reshape_Y("DOFA", Y_init.unsqueeze(0), 224, A_init.unsqueeze(0))
new_H = Y_init.shape[-1]

wavelengths_path = f"{global_path}/SS-HSU_benchmark/datasets/{dataset}_wavelength.txt"
with open(wavelengths_path, "r") as file:
    lines = file.readlines()
    wavelengths = [float(line.strip()) for line in lines if line.strip()]

In [3]:
fm_name = "DOFA"
fm, Y_init_f, new_H = rsfm.create_fm(fm_name, Y_init, size="large", version="v1", path=global_path)

_, features = rsfm.extract_f(fm, Y_init, new_H, wavelengths)
input_size = new_H
D = int(features.shape[0])
alpha = int(features.shape[1]**0.5)

### UNMamba/ CNNAEU

In [4]:
class Encoder(nn.Module):
    def __init__(self, B, c, dim, scale=3.5, group_num=4, encoder_type="UNMamba"):
        super(Encoder, self).__init__()

        if encoder_type == "UNMamba":
            self.dropout = 5e-2
            self.encoder = nn.Sequential(
                nn.Conv2d(in_channels=B, out_channels=dim, kernel_size=1, stride=1, padding=0),
                nn.GroupNorm(group_num, dim),
                nn.SiLU(),
                nn.Conv2d(in_channels=dim, out_channels=128, kernel_size=1, stride=1, padding=0),
                nn.GroupNorm(group_num, 128),
                nn.SiLU(),
                nn.Conv2d(in_channels=128, out_channels=c, kernel_size=1, stride=1, padding=0),
                nn.BatchNorm2d(c),
                unmx.Sum_to_one(scale=scale),
                nn.Dropout(self.dropout)
            )
        elif encoder_type == "CNNAEU":
            lrelu_params = {
                "negative_slope": 0.02,
                "inplace": True,
            }
            dropout = 0.2
            self.encoder = nn.Sequential(
                nn.Conv2d(B, dim, kernel_size=3, padding=1, padding_mode="zeros", bias=False),
                # nn.Conv2d(in_channels=B, out_channels=dim, kernel_size=1, stride=1, padding=0),
                nn.LeakyReLU(**lrelu_params),
                nn.BatchNorm2d(dim),
                # nn.GroupNorm(group_num, dim),
                # nn.SiLU(),
                # nn.Dropout2d(p=dropout),
                nn.Dropout(p=dropout),
                nn.Conv2d(dim, c, kernel_size=1, bias=False),
                nn.LeakyReLU(**lrelu_params),
                # nn.SiLU(),
                nn.BatchNorm2d(c),
                # nn.Dropout2d(p=dropout),
                unmx.Sum_to_one(scale=scale),
                nn.Dropout(p=dropout)
            )

        else:
            raise(f"Unknown encoder_type {encoder_type}")
    
    def forward(self, Y):
        A_hat = self.encoder(Y)
        return A_hat

class Unmixer(nn.Module):
    
    def __init__(self, B, c, dim, num_queries_times=1, encoder_type="UNMamba", decoder_type="UNMamba", loss_type="UNMamba"):
        super(Unmixer, self).__init__()
        self.decoder_type = decoder_type
        self.loss_type = loss_type
        self.B = B
        self.c = c

        self.encoder = Encoder(B, c, dim, encoder_type=encoder_type)

        if decoder_type == "UNMamba":
            self.num_queries = num_queries_times * c
            self.query_embed = nn.Embedding(self.num_queries, B)
            self.weights = nn.Parameter(torch.ones((c, num_queries_times)))
        elif decoder_type == "CNNAEU":
            self.decoder = nn.Conv2d(self.c, self.B, kernel_size=11, padding=5, padding_mode="reflect", bias=False)
        else:
            raise(f"Unknown decoder_type {decoder_type}")

        self._reset_parameters()

    def loss(self, Y_gt, Y_hat, E_hat, A_hat):
        sad = losses.SADLoss()
        mse = nn.MSELoss()

        if self.loss_type == "UNMamba":
            loss = 0
            hsi_mean = Y_gt.mean(dim=(2,3)).flatten(0).repeat(c).reshape(B, c)
            loss += mse(Y_gt, Y_hat)
            loss += sad(Y_gt, Y_hat)
            loss += mse(hsi_mean, E_hat)
            A_hat_n = torch.norm(A_hat, p=0.5, dim=1)
            loss += 1e-2 * A_hat_n.mean()
        elif self.loss_type == "CNNAEU":
            loss = sad(Y_gt, Y_hat)
        else:
            raise(f"Unknown loss_type {self.loss_type}")

        return loss

    def _reset_parameters(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight)
            elif isinstance(m, nn.GroupNorm):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                torch.nn.init.xavier_uniform(m.weight)
    
    def forward(self, Y):
        A_hat = self.encoder(Y)

        if self.decoder_type == "UNMamba":
            query_embed_weight_split = torch.chunk(self.query_embed.weight, self.c, dim=0)
            query_embed_weight_split = torch.stack(query_embed_weight_split)
            E_hat = self.weights.unsqueeze(-1).repeat(1, 1, self.B) * query_embed_weight_split
            E_hat = torch.mean(E_hat, dim=1).T
            Y_hat = torch.einsum('bchw,lc->blhw', [A_hat, E_hat])

        elif self.decoder_type == "CNNAEU":
            Y_hat = self.decoder(A_hat)
            E_hat = self.decoder.weight.detach().mean((2, 3))
        else:
            raise(f"Unknown decoder_type {self.decoder_type}")

        return Y_hat, E_hat, A_hat


In [5]:
def train(model, optimizer, loader, Y_init, E_init, A_init, epochs=200, n_xp=10):

    mses, sads = [], []

    for i in range(n_xp):
        print(f"training {i+1}/{n_xp}")

        if model.decoder_type == "CNNAEU":
            model = models.init_decoder_weights(model, Y_init/Y_init.max(), c, kernel=11)

        for _ in range(epochs):

            for Y, _, _ in loader:
            
                optimizer.zero_grad()
                Y = utils.oneD_to_2d(Y).to(dev)
                Y = rsfm.reshape_Y("DOFA", Y, 224)
                Y_hat, E_hat, A_hat = model(Y)

                loss = model.loss(Y, Y_hat, E_hat, A_hat)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=10, norm_type=1)
                optimizer.step()

                if model.decoder_type == "UNMamba":
                    for p in model.query_embed.weight:
                        p.data.clamp_(1e-7, 1)
                else:
                    with torch.no_grad():
                        constraints = unmx.Weight_constraint()
                        model.decoder.apply(constraints)
        
        model.eval()
        
        with torch.no_grad():
            
            Y_hat, E_hat, A_hat = model(Y)

            if i < (n_xp-1):
                sad, _, mse = plots.compute_metrics_and_plot(E_hat, A_hat, A_init, E_init, normalise_E=True, normalise_A=True, return_results=True, plot_E=False, plot_A=False)
            else:
                sad, _, mse = plots.compute_metrics_and_plot(E_hat, A_hat, A_init, E_init, normalise_E=True, normalise_A=True, return_results=True)
            print(f"Current SAD = {format(sad, '.3f')}, NMSE = {format(mse, '.3f')}")
            mses.append(mse)
            sads.append(sad)
    print(f"Model has {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6}M params")
    print(f"Average SAD = {format(torch.mean(torch.tensor(sads)), '.4f')} ± {format(torch.std(torch.tensor(sads)), '.3f')}, NMSE = {format(torch.mean(torch.tensor(mses)), '.4f')} ± {format(torch.std(torch.tensor(mses)), '.3f')}")

# model = Unmixer(B, c, dim=48, num_queries_times=50, encoder_type="UNMamba", decoder_type="UNMamba", loss_type="UNMamba")
# lr = 0.02 #3e-3
# optimizer = torch.optim.Adam(model.parameters(), lr)
# # optimizer = torch.optim.AdamW(model.parameters(), lr, weight_decay=0.2)
# loader, _, _ = utils.create_dataloader(dataset, patch_size=H, dev=dev, path=global_path)
# train(model, optimizer, loader, Y_init, E_init, A_init, epochs=300)

### HFM

In [6]:
class HFMUnmixer(nn.Module):
    def __init__(self, B, c, D, alpha, H, dim=48, num_queries_times=50, group_num=4, scale=3.5, dropout=5e-2):
        super(HFMUnmixer, self).__init__()
        self.B = B
        self.c = c
        self.H = H

        self.upsampler = nn.Linear(alpha**2, H**2)
        self.dropout = 5e-2
        self.abundance_estimator = nn.Sequential(
            nn.Conv2d(in_channels=D, out_channels=dim, kernel_size=1, stride=1, padding=0),
            nn.GroupNorm(group_num, dim),
            nn.SiLU(),
            nn.Conv2d(in_channels=dim, out_channels=128, kernel_size=1, stride=1, padding=0),
            nn.GroupNorm(group_num, 128),
            nn.SiLU(),
            nn.Conv2d(in_channels=128, out_channels=c, kernel_size=1, stride=1, padding=0),
            nn.BatchNorm2d(c),
            unmx.Sum_to_one(scale=scale),
            nn.Dropout(self.dropout)
        )

        # self.abundance_estimator = nn.Sequential(
        #     nn.Conv2d(D, c, kernel_size=1, bias=False, padding="same"),
        #     nn.LeakyReLU(0.02),
        #     nn.BatchNorm2d(c),
        #     nn.Dropout(5e-2)
        # )

        self.num_queries = num_queries_times * c
        self.query_embed = nn.Embedding(self.num_queries, B)
        self.weights = nn.Parameter(torch.ones((c, num_queries_times)))

        self._reset_parameters()

    def _reset_parameters(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight)
            elif isinstance(m, nn.GroupNorm):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                torch.nn.init.xavier_uniform_(m.weight)

    def loss(self, Y_gt, Y_hat, E_hat, A_hat):
        sad = losses.SADLoss()
        mse = nn.MSELoss()

        loss = 0
        hsi_mean = Y_gt.mean(dim=(2,3)).flatten(0).repeat(c).reshape(B, c)
        loss += mse(Y_gt, Y_hat)
        loss += sad(Y_gt, Y_hat)
        loss += mse(hsi_mean, E_hat)
        A_hat_n = torch.norm(A_hat, p=0.5, dim=1)
        loss += 1e-2 * A_hat_n.mean()
        # loss = sad(Y_gt, Y_hat)
        return loss
    
    # def loss(loss, Y_gt, Y_hat, A_hat, E_hat, W_sad=1, W_ab=0.6, W_tv_e=3e-5, W_mse=0.09):
    #     sad = losses.SADLoss()
    #     mse = nn.MSELoss(reduction='sum')
    #     mse2 = nn.MSELoss()
        
    #     loss_sad = W_sad * sad(Y_gt, Y_hat)
    #     loss_ab = W_ab * torch.sqrt(A_hat).mean()

    #     loss_mse = W_mse * mse(Y_gt, Y_hat)/(torch.norm(Y_gt)**2)

    #     """Abundances and endmembers regularisation"""

    #     # TV on endmembers (sum of difference between consecutive endmembers)
    #     loss_tv_e = W_tv_e * (torch.abs(E_hat[:, 1:] - E_hat[:, :-1]).sum())

    #     # Endmember regul : distance to mean spectrum
    #     # B, c = E_hat.shape
    #     # mean_e = Y_gt.mean(dim=(2,3)).flatten(0).repeat(c).reshape(B, c)
    #     # loss_e = W_e * mse2(mean_e, E_hat)
    #     # loss_a = W_a * torch.norm(A_hat, p=0.5, dim=1).mean()

    #     loss = loss_sad + loss_ab + loss_tv_e + loss_mse

    #     return loss

    def get_endmembers(self):
        # E_hat = self.decoder.weight.detach().mean((2, 3))
        query_embed_weight_split = torch.chunk(self.query_embed.weight, self.c, dim=0)
        query_embed_weight_split = torch.stack(query_embed_weight_split)
        E_hat = self.weights.unsqueeze(-1).repeat(1, 1, self.B) * query_embed_weight_split
        E_hat = torch.mean(E_hat, dim=1).T
        return E_hat

    def forward(self, feat):
        feat_up = self.upsampler(feat)
        feat_up = utils.oneD_to_2d(feat_up)
        feat_up = utils.standardise(feat_up).unsqueeze(0)
        A_hat = self.abundance_estimator(feat_up)
        E_hat = self.get_endmembers()
        Y_hat = torch.einsum('Bchw,bc->Bbhw', [A_hat, E_hat])

        return Y_hat, E_hat, A_hat

In [7]:
def train(model, fm, optimizer, loader, Y_init, E_init, A_init, epochs=200, n_xp=5):

    mses, sads = [], []
    # E_hats, A_hats = torch.zeros(n_xp, model.B, model.c), torch.zeros(n_xp, model.c, model.H, model.H)

    for i in range(n_xp):
        print(f"training {i+1}/{n_xp}")

        # if model.decoder_type == "CNNAEU":
        # model = models.init_decoder_weights(model, Y_init/Y_init.max(), model.c, kernel=1)

        for _ in range(epochs):

            for Y, _, _ in loader:
            
                optimizer.zero_grad()
                Y = utils.oneD_to_2d(Y).to(dev)
                Y = rsfm.reshape_Y(fm_name, Y, 224)
                _, feat = rsfm.extract_f(fm, Y, new_H, wavelengths)
                Y_hat, E_hat, A_hat = model(feat)

                loss = model.loss(Y, Y_hat, E_hat, A_hat)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=10, norm_type=1)
                optimizer.step()

                for p in model.query_embed.weight:
                    p.data.clamp_(1e-7, 1)
                # with torch.no_grad():
                #     constraints = unmx.Weight_constraint()
                #     model.decoder.apply(constraints)
        
        model.eval()
        
        with torch.no_grad():
        
            _, features = rsfm.extract_f(fm, Y, new_H, wavelengths)
            _, E_hat, A_hat = model(features)

            if i < (n_xp-1):
                sad, _, mse = plots.compute_metrics_and_plot(E_hat, A_hat, A_init, E_init, normalise_E=True, normalise_A=True, return_results=True, plot_E=False, plot_A=False)
            else:
                sad, _, mse = plots.compute_metrics_and_plot(E_hat, A_hat, A_init, E_init, normalise_E=True, normalise_A=True, return_results=True)
            print(f"Current SAD = {format(sad, '.3f')}, NMSE = {format(mse, '.3f')}")
            mses.append(mse)
            sads.append(sad)
        # E_hats[i] = E_hat
        # A_hats[i] = A_hat
    print(f"Model has {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6}M params")
    print(f"Average SAD = {format(torch.mean(torch.tensor(sads)), '.4f')} ± {format(torch.std(torch.tensor(sads)), '.3f')}, NMSE = {format(torch.mean(torch.tensor(mses)), '.4f')} ± {format(torch.std(torch.tensor(mses)), '.3f')}")

    # E_hat_m = torch.nanmean(E_hats)
    # A_hat_m = torch.nanmean(A_hats)
    # print(E_hat_m.shape, A_hat_m.shape)
    # sad, _, mse = plots.compute_metrics_and_plot(E_hat_m, A_hat_m, A_init, E_init, normalise_E=True, normalise_A=True, return_results=True)
    # print(f"Mean SAD = {format(sad, '.4f')}, NMSE = {format(mse, '.4f')}")

model = HFMUnmixer(B, c, D, alpha, new_H, dim=48, num_queries_times=50)
lr = 0.02 #3e-3
# optimizer = torch.optim.Adam(model.parameters(), lr)
optimizer = torch.optim.AdamW(model.parameters(), lr, weight_decay=0.18)
loader, _, _ = utils.create_dataloader(dataset, patch_size=H, dev=dev, path=global_path)
train(model, fm, optimizer, loader, Y_init, E_init, A_init, epochs=300)

training 1/5
Current SAD = 0.373, NMSE = 1.479
training 2/5


KeyboardInterrupt: 